In [1]:
# imports

import sys
import ast
import html
import pandas as pd
import numpy as np

# find text_functions.py in the src folder
sys.path.append('../src')
from text_functions import review_cleaner, department_group, grade_points, leniency_pos, leniency_neg, teaching_pos, teaching_neg

##### Load raw data

In [2]:
# load the merged, raw dataset
reviews_df = pd.read_csv('../data/interim/rmp_raw.csv')

print("Reviews:", len(reviews_df))
print(reviews_df['school'].value_counts())

Reviews: 57924
school
rit          12080
brockport    12065
geneseo      12030
rochester    12002
fisher        5140
nazareth      4607
Name: count, dtype: int64


##### Remove reviews without real text, and duplicates

In [3]:
before = len(reviews_df)

# drop missing, blank, and placeholder comments
reviews_df = reviews_df.dropna(subset=['comment'])
reviews_df = reviews_df[reviews_df['comment'].str.strip() != '']
reviews_df = reviews_df[reviews_df['comment'].str.strip() != 'No Comments']
after_text = len(reviews_df)

# no RMP review id, so a duplicate is the same professor, date, and comment
reviews_df = reviews_df.drop_duplicates(subset=['professor_id', 'date', 'comment'])

print("Removed for no comment:", before - after_text)
print("Removed as duplicates:", after_text - len(reviews_df))
print("Kept:", len(reviews_df))

Removed for no comment: 1290
Removed as duplicates: 37
Kept: 56597


##### Fix the comment text, dates, and other columns

In [4]:
# turn html codes like back into normal characters
reviews_df['comment'] = reviews_df['comment'].apply(html.unescape)

# dates are already YYYY-MM-DD, convert them and pull out the year
reviews_df['date'] = pd.to_datetime(reviews_df['date'])
reviews_df['year'] = reviews_df['date'].dt.year

# positive review (4 or higher) used for classification
reviews_df['positive'] = (reviews_df['quality'] >= 4).astype(int)

# letter grade as a number (non grades become missing)
reviews_df['grade_points'] = reviews_df['grade'].map(grade_points)

# combine the two attendance formats
reviews_df['attendance'] = reviews_df['attendance'].replace({'Y': 'mandatory', 'N': 'non mandatory'})

# -1 means the student did not answer the textbook question
reviews_df['textbook'] = reviews_df['textbook'].replace(-1, np.nan)

# tags are a list saved as text, turn them back into a real list and lowercase them
reviews_df['tags'] = reviews_df['tags'].apply(ast.literal_eval)
reviews_df['tags'] = reviews_df['tags'].apply(lambda tags: [tag.lower() for tag in tags])

# check the values after cleaning
print(reviews_df['grade_points'].value_counts(dropna=False))
print(reviews_df['attendance'].value_counts(dropna=False))

grade_points
NaN    34280
4.0    11723
3.7     3318
3.3     2415
3.0     2243
2.7      859
2.0      600
2.3      510
1.7      251
0.0      136
1.0      132
1.3       89
0.7       41
Name: count, dtype: int64
attendance
NaN              27025
mandatory        18914
non mandatory    10658
Name: count, dtype: int64


##### Group departments

In [5]:
# assign each department to Social Science, STEM, Humanities, or Other
reviews_df['dept_group'] = reviews_df['department'].apply(department_group)
print(reviews_df['dept_group'].value_counts())

# departments that did not match any group, add these to the lists in text_functions.py if needed
print(reviews_df.loc[reviews_df['dept_group'] == 'Other', 'department'].value_counts().head(30))

# quick test before running on every review
examples = ["He is not very clear and the exams aren't easy.",
            "The homework isn't super easy but he explains everything clearly.",
            "Never curves. Tough grader but lectures were engaging."]
for example in examples:
    print(review_cleaner(example))

dept_group
STEM              22133
Social Science    17576
Humanities        16064
Other               824
Name: count, dtype: int64
department
Delta                             117
Freshman Year Experience           98
Not Specified                      80
Dance                              74
Networking Security & Systems      65
Networking Security & Systems      58
Interactive Games & Media          49
Honors                             43
Recreation                         37
Biotechnology                      30
Networking Security  Systems       29
Professional Studies               22
Interactive Games amp Media        19
Pharmacy                           16
Nutrition                          14
Professional Programs              11
Photography                        10
Architecture                        8
Public Administration               8
Meteorology                         8
Athletics                           7
Law                                 6
Photo Illustration  

##### Clean every review

In [6]:
# clean and lemmatize every review
reviews_df['review_clean'] = reviews_df['comment'].apply(review_cleaner)

# reviews with nothing left after cleaning
print("Empty after cleaning:", (reviews_df['review_clean'] == '').sum())
reviews_df[['comment', 'review_clean']].head(10)

# cleaned word lists used for the leniency and teaching scores
print("Leniency +: ", leniency_pos)
print("Leniency -: ", leniency_neg)
print("Teaching +: ", teaching_pos)
print("Teaching -: ", teaching_neg)

# save the cleaned dataset for scoring and analysis
reviews_df.to_csv('../data/interim/rmp_clean.csv', index=False)
print("Saved", reviews_df.shape)

Empty after cleaning: 51
Leniency +:  ['easy', 'easy grader', 'easy class', 'easy exam', 'easy grade', 'lenient', 'generous', 'generous grader', 'curve', 'extra credit', 'credit opportunity', 'partial credit', 'bonus point', 'easy point', 'free point', 'drop low', 'grade boost', 'chill', 'light workload', 'not_harsh', 'not_harshly', 'not_strict', 'not_picky', 'not_nitpicky', 'not_unreasonable', 'not_unfair', 'not_subjective', 'not_deducts']
Leniency -:  ['harsh', 'harshly', 'grade harshly', 'harsh grader', 'harsh grade', 'tough grader', 'hard grader', 'strict', 'strict grader', 'picky', 'nitpicky', 'unreasonable', 'unfair', 'subjective', 'inconsistent grade', 'deducts', 'take point', 'lose point', 'low average', 'class average', 'grade deflation', 'impossible get', 'not_easy', 'not_lenient', 'not_generous', 'not_curve', 'not_chill']
Teaching +:  ['clear', 'clearly', 'explains', 'explains well', 'explains concept', 'great lecturer', 'amazing lecture', 'great lecture', 'engage', 'engage 